In [ ]:

# This notebook loads a custom fine-tuned language model and applies it to medical texts to extract entities and scores automatically. 
# The results are saved for further analysis or evaluation.


import torch
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments, TextStreamer
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported

from pydantic import BaseModel
from typing import Optional, List, Union
from pydantic import Field
import json
from collections import OrderedDict, defaultdict, Counter

import pickle
from datasets import Dataset
import pandas as pd
import dspy
import csv
import re

import ast
import tqdm
from collections import OrderedDict

from json_repair import repair_json

from huggingface_hub import login

In [ ]:
refined ="models/evaluation/final_model" 

max_seq_length = 15000 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, tokenizer = FastLanguageModel.from_pretrained("models/evaluation/final_model", max_seq_length )

In [ ]:
model = FastLanguageModel.for_inference(model)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)


In [ ]:
import os
import json

folder = 'data/json/split/test'
files = os.listdir(folder)

all_data = []
for filename in files:
    filepath = os.path.join(folder, filename)
    if os.path.isfile(filepath):  # Nur Dateien öffnen!
        with open(filepath) as f:
            dataset_test = json.load(f)
            all_data.append(dataset_test)

print(f"{len(all_data)} Dateien geladen.")


In [ ]:
# Replace with the path to your folder
folder_path = 'data/json/split/test'

# Initialize list to store parsed data
test_dict_data = []

# Loop through JSON files and extract required fields
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    
    # Ensure it's a file and ends with .json
    if os.path.isfile(file_path) and filename.endswith('.json'):
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)

            # Access "text" and "annotations" keys
            text = data.get('text')
            annotations = data.get('annotations', [])

            # Remove 'begin' and 'end' keys from each annotation dict
            cleaned_annotations = [
                {k: v for k, v in annotation.items() if k not in ('begin', 'end')and v not in ('', None)}
                for annotation in annotations
            ]

            # Append to list in desired format
            test_dict_data.append({
                'filename': filename,
                'prompt': text,
                'response': str(cleaned_annotations)
            })
        
for item in test_dict_data[:2]:
     print(item)

In [ ]:
import os
import json
import csv

folder_path = 'data/json/split/test'

test_dict_data = []

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path) and filename.endswith('.json'):
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)

            text = data.get('text')
            annotations = data.get('annotations', [])
            cleaned_annotations = [
                {k: v for k, v in annotation.items() if k not in ('begin', 'end') and v not in ('', None)}
                for annotation in annotations
            ]
            
            test_dict_data.append({
                'filename': filename,
                'prompt': text,
                'response': str(cleaned_annotations)
            })

for item in test_dict_data[:2]:
     print(item)
     
with open('output_all_letters.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile, delimiter=';')
    writer.writerow(['filename', 'response'])
    for item in test_dict_data:
        writer.writerow([item['filename'], item['response']])

print(f"{len(test_dict_data)} Briefe wurden verarbeitet und gespeichert!")


In [ ]:
# Optional: print or process the final list
for item in test_dict_data[:1]:
     print(item)

In [ ]:
test_dataset = Dataset.from_pandas(pd.DataFrame(test_dict_data))

In [ ]:
class CardioScoreAnnotation(BaseModel):
    reasoning: Optional[str]
    entity_type: Optional[str]
    typ: Optional[str]
    aktualitaet: Optional[str]
    numeral: Optional[float]
    unit: Optional[str]
    #scores: Optional[str]
    annotated_span: str

# Document model
class CardioScoreDocument(BaseModel):
    text: str
    annotations: List[CardioScoreAnnotation]

In [ ]:
schema = json.dumps(CardioScoreDocument.model_json_schema())

# Define system message
system_message = f"""You are a physician. Your task is to extract medication information with entity_types:

Alter, Antithrombotic treatment, Bempedoinsäure, Blutdruck (systolisch) (RR), Cholesterin, Diabetes,  Mellitus, Ezetimib, Gefäßerkrankung, Geschlecht, HDL Cholesterin, HbA1c, Herzinsuffizienz, Hypertonie, LDL Cholesterin, Nikotinkonsum, SGLT2 Inhibitoren, Schlaganfall/TIA, Scores, Scores (explicit), Statinse, GFR CKD-EPI

from a given German doctoral letter from cardiology. In addition extract the related information for reasoning, typ, aktualitaet, numeral, unit, scores IF applicable.

If there is NO medication information for an entity_type IGNORE it.
Ignore the <[Pseudo] > tags aroung dates, e.g. <[Pseudo] 2000-04-09 > is 2000-04-09.
ONLY respond with an instance of JSON without any additional information. You have access to a JSON schema, which will determine how the JSON should be structured.
Make sure to return ONLY an instance of the JSON, NOT the schema itself. Do not add any additional information.
JSON schema:
{schema}
"""
print(schema)

In [ ]:
# Convert into chat template
def create_conversation(sample):
    # Create conversation with system, user, and assistant
    conversation = {
        "filename": sample["filename"],  # System message
        "messages": [
            {"role": "system", "content": system_message},  # System message
            {"role": "user", "content": sample["prompt"]},  # User prompt
            {"role": "assistant", "content": sample["response"]}  # Assistant's expected response
        ]
        
    }
    return conversation
    
# Apply the function to the dataset
test_chat_dataset = test_dataset.map(create_conversation, remove_columns=['prompt', 'response'])


In [ ]:
def apply_template(examples):
    messages = examples["messages"]
    text = [tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=False) for message in messages]
    return {"text": text}

final_test = test_chat_dataset.map(apply_template, batched=True)

In [ ]:
# --- Schritt 1: Die 3 Predictions generieren ---
predictions = {}
for i in range(49):
    letter = final_test['messages'][i]
    filename = final_test['filename'][i]

    user_text = "\n\n".join([msg['content'] for msg in letter if msg['role'] == "user"])
    system_msg = next((msg['content'] for msg in letter if msg['role'] == "system"), system_message)

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_text},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=4024,
        use_cache=True,
        do_sample=False,
        temperature=1.0
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    assistant_response = decoded
    if "<|im_start|>assistant" in decoded:
        assistant_response = decoded.split("<|im_start|>assistant", 1)[1]
        if "<|im_end|>" in assistant_response:
            assistant_response = assistant_response.split("<|im_end|>", 1)[0]
        assistant_response = assistant_response.strip()

    predictions[filename] = assistant_response

    print("Filename:", filename, "...")
    
    
# --- Schritt 2: Die bestehende CSV um Predictions ergänzen ---
input_csv = 'output_all_letters.csv'
output_csv = 'output_all_letters_with_pred.csv'

rows = []
with open(input_csv, 'r', encoding='utf-8') as infile:
    reader = csv.DictReader(infile, delimiter=';')
    fieldnames = reader.fieldnames + ['prediction'] if 'prediction' not in reader.fieldnames else reader.fieldnames
    for row in reader:
        # Schreibe Prediction nur für die ersten 3 Briefe
        if row['filename'] in predictions:
            row['prediction'] = predictions[row['filename']]
        else:
            row['prediction'] = ''
        rows.append(row)

with open(output_csv, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter=';')
    writer.writeheader()
    writer.writerows(rows)

print("Die Predictions für die ersten 3 Briefe wurden gespeichert:", output_csv)

# Evaluation


In [ ]:
import pandas as pd
import ast
import numpy as np

# --- Datei laden ---
df = pd.read_csv("all_letters_with_pred_entity_type.csv", sep=",", encoding="utf-8")

# --- Extraktions-Logik ---
def extract_vals(cell, key):
    if pd.isna(cell) or cell is np.nan:
        return []
    values = []
    for part in str(cell).split('|'):
        part = part.strip()
        if not part:
            continue
        # Dict-Parsing
        if part.startswith("{") and key in part:
            try:
                d = ast.literal_eval(part)
                if isinstance(d, dict) and key in d:
                    values.append(str(d[key]))
            except:
                continue
        # Key-Value-Parsing
        elif f"{key}:" in part:
            try:
                for item in part.split(";"):
                    if f"{key}:" in item:
                        value = item.split(f"{key}:")[1].strip()
                        values.append(value)
            except:
                continue
    return values

def extract_nikotinkonsum(cell):
    if pd.isna(cell) or cell is np.nan:
        return []
    values = []
    for part in str(cell).split('|'):
        part = part.strip()
        if "aktualitaet:" in part:
            try:
                value = part.split("aktualitaet:")[1].strip()
                values.append(value)
            except:
                continue
        elif "typ:" in part:
            try:
                value = part.split("typ:")[1].strip()
                values.append(value)
            except:
                continue
    return values

def extract_scores(cell):
    if pd.isna(cell) or cell is np.nan:
        return []
    values = []
    for part in str(cell).split('|'):
        part = part.strip()
        if "scores:" in part:
            try:
                value = part.split("scores:")[1].strip()
                values.append(value)
            except:
                continue
        elif "numeral:" in part:
            try:
                value = part.split("numeral:")[1].strip()
                values.append(value)
            except:
                continue
    return values

# --- Eval-Funktionen ---
def eval_typ(row):
    gold = extract_vals(row["Gold-Standard"], "typ")
    pred = extract_vals(row["Prediction"], "typ")
    if not gold and not pred:
        return ""
    elif gold and not pred:
        return "FN"
    elif not gold and pred:
        return "FP"
    elif any(g in p or p in g for g in gold for p in pred):
        return "TP"
    else:
        return "FP"

def eval_numeral(row):
    gold = extract_vals(row["Gold-Standard"], "numeral")
    pred = extract_vals(row["Prediction"], "numeral")
    if not gold and not pred:
        return ""
    elif gold and not pred:
        return "FN"
    elif not gold and pred:
        return "FP"
    elif any(g == p for g in gold for p in pred):
        return "TP"
    else:
        return "FP"

def eval_entity_type(row):
    gold = extract_vals(row["Gold-Standard"], "entity_type")
    pred = extract_vals(row["Prediction"], "entity_type")
    if not gold and not pred:
        return ""
    elif gold and not pred:
        return "FN"
    elif not gold and pred:
        return "FP"
    elif any(g == p for g in gold for p in pred):
        return "TP"
    else:
        return "FP"

def eval_nikotinkonsum(row):
    gold = extract_nikotinkonsum(row["Gold-Standard"])
    pred = extract_nikotinkonsum(row["Prediction"])
    if not gold and not pred:
        return ""
    elif gold and not pred:
        return "FN"
    elif not gold and pred:
        return "FP"
    elif any(g == p for g in gold for p in pred):
        return "TP"
    else:
        return "FP"

def eval_scores(row):
    gold = extract_scores(row["Gold-Standard"])
    pred = extract_scores(row["Prediction"])
    if not gold and not pred:
        return ""
    elif gold and not pred:
        return "FN"
    elif not gold and pred:
        return "FP"
    elif any(g == p for g in gold for p in pred):
        return "TP"
    else:
        return "FP"

# --- Eval-Spalten berechnen ---
df["typ_eval"] = df.apply(eval_typ, axis=1)
df["numeral_eval"] = df.apply(eval_numeral, axis=1)
df["entity_type_eval"] = df.apply(eval_entity_type, axis=1)
df["nikotinkonsum_eval"] = df.apply(eval_nikotinkonsum, axis=1)
df["scores_eval"] = df.apply(eval_scores, axis=1)

for col in ["typ_eval", "numeral_eval", "entity_type_eval", "nikotinkonsum_eval", "scores_eval"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

# --- Mapping Entity-Type → Eval-Spalte ---
entity_type_map = {
    "Statins": "typ_eval",
    "Antithrombotic treatment": "typ_eval",
    "Geschlecht": "typ_eval",
    "Gefäßerkrankung": "typ_eval",
    "Gefäßerkankung": "typ_eval",
    "Alter": "numeral_eval",
    "Blutdruck (systolisch) (RR)": "numeral_eval",
    "eGFR CKD-EPI": "numeral_eval",
    "Hypertonie": "entity_type_eval",
    "Herzinsuffizienz": "entity_type_eval",
    "Diabetes Mellitus": "entity_type_eval",
    "Nikotinkonsum": "nikotinkonsum_eval",
    "HbA1c": "numeral_eval",
    "Cholesterin": "numeral_eval",
    "Schlaganfall/TIA": "typ_eval",
    "LDL Cholesterin": "numeral_eval",
    "HDL Cholesterin": "numeral_eval",
}

entity_types = df["Entity-Type"].unique()
result = []

for ent in entity_types:
    if ent == "Scores (explicit)":
        continue

    eval_col = entity_type_map.get(ent, "typ_eval")
    df_ent = df[df["Entity-Type"] == ent]
    tp = (df_ent[eval_col] == "TP").sum()
    fp = (df_ent[eval_col] == "FP").sum()
    fn = (df_ent[eval_col] == "FN").sum()

    # Support zählen als Summe der Einträge im Gold-Standard für dieses Attribut
    if eval_col == "typ_eval":
        support = df_ent["Gold-Standard"].apply(lambda x: len(extract_vals(x, "typ"))).sum()
    elif eval_col == "numeral_eval":
        support = df_ent["Gold-Standard"].apply(lambda x: len(extract_vals(x, "numeral"))).sum()
    elif eval_col == "entity_type_eval":
        support = df_ent["Gold-Standard"].apply(lambda x: len(extract_vals(x, "entity_type"))).sum()
    elif eval_col == "nikotinkonsum_eval":
        support = df_ent["Gold-Standard"].apply(lambda x: len(extract_nikotinkonsum(x))).sum()
    elif eval_col == "scores_eval":
        support = df_ent["Gold-Standard"].apply(lambda x: len(extract_scores(x))).sum()
    else:
        support = len(df_ent)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    result.append({
        "Entity-Type": ent,
        "precision": round(precision, 2),
        "recall":    round(recall,    2),
        "f1-score":  round(f1,        2),
        "support":   support,
        "TP": tp,
        "FP": fp,
        "FN": fn
    })

result_df = pd.DataFrame(result)
print("\n=== Ergebnis-Tabelle ===\n")
print(result_df)

# Optional: Speichern
result_df.to_csv("classification_report_entity_types.csv", sep=";", index=False, encoding="utf-8")


In [ ]:
import os
import json
import pandas as pd
import ast

# ----- 1. Load predictions (main table with all data) -----
df = pd.read_csv("all_letters_with_pred_entity_type.csv", sep=",")

# ----- 2. For each row, read the corresponding JSON and extract the 'scores' field -----
json_folder = 'data/json/split/test'

def get_scores_from_json(filename):
    filepath = os.path.join(json_folder, filename)
    if not os.path.exists(filepath):
        print(f"Datei nicht gefunden: {filename}")
        return None
    try:
        with open(filepath, encoding='utf-8') as f:
            data = json.load(f)
            return data['annotations'][0]['scores']
    except Exception as e:
        print(f"Fehler beim Lesen von {filename}: {e}")
        return None
# Create a new column with the "gold standard" CHADS-VASc score from the reference JSON files

df['chadsvasc_gold'] = df['Datei'].apply(get_scores_from_json)

# ----- 3. Parse predictions and extract relevant information for scoring -----
def parse_prediction(pred_str):
    if pd.isnull(pred_str):
        return {}
    try:
        result = ast.literal_eval(pred_str)
        if isinstance(result, dict):
            return result
    except Exception:
        pass
    result = {}
    for part in str(pred_str).split('|'):
        part = part.strip()
        if ':' in part:
            key, value = part.split(':', 1)
            key = key.strip()
            value = value.strip().strip("'\"")
            try:
                value = float(value)
                if value.is_integer():
                    value = int(value)
            except Exception:
                pass
            result[key] = value
    return result

def get_score_parameters(group):
    params = {
        'Congestive heart failure/LV dysfunction': False,
        'Hypertension': False,
        'Age ≥75y': False,
        'Age 65-74y': False,
        'Diabetes mellitus': False,
        'Stroke/TIA/TE': False,
        'Vascular diseases': False,
        'Sex category': False
    }
    for _, row in group.iterrows():
        etype = row['Entity-Type']
        prediction = parse_prediction(row['Prediction'])
        if etype == 'Herzinsuffizienz' and prediction:
            params['Congestive heart failure/LV dysfunction'] = True
        elif etype == 'Hypertonie' and prediction:
            params['Hypertension'] = True
        elif etype == 'Alter' and prediction:
            alter = prediction.get('numeral', 0)
            params['Age ≥75y'] = alter >= 75
            params['Age 65-74y'] = 65 <= alter < 75
        elif etype == 'Diabetes Mellitus' and prediction:
            params['Diabetes mellitus'] = True
        elif etype == 'Schlaganfall' and prediction:
            params['Stroke/TIA/TE'] = True
        elif etype == 'Gefäßerkrankung' and prediction:
            gef_typ = prediction.get('typ', '').lower()
            params['Vascular diseases'] = gef_typ in ['khk', 'pavk', 'aortenplaque']
        elif etype == 'Geschlecht' and prediction:
            geschlecht = prediction.get('typ', '').lower()
            params['Sex category'] = geschlecht == 'weiblich'
    return params

# Points table for CHADS-VASc score calculation
POINTS = {
    'Congestive heart failure/LV dysfunction': 1,
    'Hypertension': 1,
    'Age ≥75y': 2,
    'Diabetes mellitus': 1,
    'Stroke/TIA/TE': 2,
    'Vascular diseases': 1,
    'Age 65-74y': 1,
    'Sex category': 1
}
def calc_chads_vasc_score(parameters):
    if parameters['Age ≥75y'] and parameters['Age 65-74y']:
        raise ValueError('Not both age parameters can be true!')
    return sum([bool(value) * POINTS[parameter] for parameter, value in parameters.items()])

# Calculate predicted score for each document
scores = {}
for fn, group in df.groupby('Datei'):
    params = get_score_parameters(group)
    try:
        scores[fn] = calc_chads_vasc_score(params)
    except:
        scores[fn] = ''

df['chadsvasc_pred'] = df['Datei'].map(scores)

# ----- 4. Save result -----
df.to_csv("output_with_both_scores.csv", sep=";", index=False)
print("Fertig! Datei gespeichert als: output_with_both_scores.csv")


In [ ]:
import os
import json
import collections
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Define paths to your data folders
# ------------------------------------------------------------
TEST_FOLDER  = 'data/json/split/test'
TRAIN_FOLDER = 'data/json/split/train'

# ------------------------------------------------------------
# 2. Specify which key contains the entity type in your annotations.
#    -> This can vary by dataset, e.g., "entity_type", "label", "type", or "name".
#    -> Adjust the tuple below if necessary for your JSON structure.
# ------------------------------------------------------------
ENTITY_TYPE_KEYS = ("entity_type", "label", "type", "name")

# ------------------------------------------------------------
# 3. Helper function: Count entity types for all files in a folder
# ------------------------------------------------------------
def count_entities_in_folder(folder: str) -> collections.Counter:
    counter = collections.Counter()
    for fn in os.listdir(folder):
        if not fn.endswith('.json'):
            continue
        fp = os.path.join(folder, fn)
        try:
            with open(fp, encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print(f"[WARN] {fn}: {e}")
            continue

        # Each JSON may have multiple annotations
        for ann in data.get("annotations", []):
            etype = None
            # Find the key which holds the entity type
            for key in ENTITY_TYPE_KEYS:
                if key in ann:
                    etype = ann[key]
                    break
            if etype:
                counter[etype] += 1
    return counter

# ------------------------------------------------------------
# 4. Count entities in both train and test folders
# ------------------------------------------------------------
test_counts  = count_entities_in_folder(TEST_FOLDER)
train_counts = count_entities_in_folder(TRAIN_FOLDER)

# ------------------------------------------------------------
# 5. Combine counts into a single DataFrame for analysis
# ------------------------------------------------------------
all_entity_types = sorted(set(test_counts) | set(train_counts))
df = pd.DataFrame(
    {
        "train": [train_counts.get(et, 0) for et in all_entity_types],
        "test":  [test_counts.get(et, 0)  for et in all_entity_types],
    },
    index=all_entity_types,
)
df["total"] = df["train"] + df["test"]
df.sort_values("total", ascending=False, inplace=True)

print("\n=== Entity-Typ-Häufigkeiten ===")
print(df)

# Optionally, save as CSV for further analysis/sharing
df.to_csv("entity_type_counts.csv")
print("\n→ Tabelle gespeichert als entity_type_counts.csv")

# ------------------------------------------------------------
# 6. Optional: Bar plot comparing train and test distributions
# ------------------------------------------------------------
PLOT = True  
if PLOT:
    ax = df[["train", "test"]].plot(
        kind="bar",
        figsize=(12, 6),
        title="Entity-Typ-Verteilung in Train vs. Test",
    )
    ax.set_xlabel("Entity-Typ")
    ax.set_ylabel("Anzahl")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import json
import pandas as pd
from IPython.display import display

# ─── 1) Point to train JSONs  ───
train_folder = 'data/json/split/train'

# ─── 2) Prepare containers ───
records_herz = []
records_diab = []

# ─── 3) Loop over files and collect spans ───
for fname in os.listdir(train_folder):
    if not fname.endswith('.json'):
        continue
    fp = os.path.join(train_folder, fname)
    with open(fp, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for ann in data.get('annotations', []):
        et = ann.get('entity_type')
        span = ann.get('annotated_span')
        if et == 'Herzinsuffizienz':
            records_herz.append({'filename': fname, 'annotated_span': span})
        elif et == 'Diabetes Mellitus':
            records_diab.append({'filename': fname, 'annotated_span': span})

# ─── 4) Build DataFrames ───
df_herz = pd.DataFrame(records_herz)
df_diab = pd.DataFrame(records_diab)

# ─── 5) Display them ───
print("Herzinsuffizienz spans:")
display(df_herz)

print("Diabetes Mellitus spans:")
display(df_diab)
